In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('propanol_production_monte_carlo')

In [4]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'inventories-ei310']

In [5]:
methods = [('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
len(methods)

1

In [6]:
methods

[('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]

In [7]:
db = bd.Database('inventories-ei310')
acts = [act for act in db if 'propanol production' in act['name'] and 'GLO' == act['location']]
len(acts)

5

In [8]:
mc_df = pd.DataFrame()

In [9]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(500):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [10]:
mc_df

,"propanol production, fossil","propanol production, biogas, green ethylene","propanol production, biogas, fossil ethylene","propanol production, DAC carbon dioxide, wind hydrogen, green ethylene","propanol production, DAC carbon dioxide, wind hydrogen, fossil ethylene"
0,6.439727,-0.384447,1.724687,0.041732,2.154353
1,5.683735,-0.335370,1.627684,-0.101666,1.864633
2,5.190692,-0.504566,1.352938,-0.134592,1.725982
3,5.444226,-0.713861,1.431704,-0.473590,1.675522
4,5.547061,0.082393,1.862169,0.765246,2.547964
...,...,...,...,...,...
495,5.494841,-0.534643,1.866152,-0.181929,2.222835
496,4.599357,-0.673849,1.253729,-0.447976,1.482788
497,4.983698,-0.597091,1.527923,-0.355657,1.772870
498,6.273290,-0.070380,1.481269,0.418529,1.972743


In [11]:
mc_df.to_excel(os.path.join('..', 'results', 'climate_change_monte_carlo.xlsx'))